# Experimento: Ejecutar HPT con Búsqueda en Cuadrícula para construir un modelo GBT

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import string
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize

from src import utils

# Parámetros

In [3]:
RND_SEED = 123
PCT_TEST = 0.2
K_FOLD = 3

EXPERIMENT = "exp02_hpt_gbt"

# Paths
path_interim = os.path.join("data", "interim")
path_experiment =  os.path.join(path_interim, EXPERIMENT)

# Input
file_train = "train.csv"


# Output
file_exp = "df_exp_summary.csv"

In [4]:
utils.create_or_clean_folder(path_experiment)


Creating the folder: data\interim\exp02_hpt_gbt


# Cargar datos

In [5]:
path_data_train = os.path.join(path_interim, file_train)

df_train = pd.read_csv(path_data_train)
df_train.head()

,x_text,y_is_nf
0,Respuestas coherentes e idénticas ante entrada...,0
1,Gestión de usuarios: Todos los administradores...,0
2,Añadir numero de una revista. Para ello debemo...,0
3,Un usuario registrado visualiza la tabla de en...,0
4,Como usuario quiero poder ordenar las listas d...,0


# Construir Tubería

In [6]:
# Celda auxiliar: Tokenización y lematización en español
import typing
import string


def tokenizer_stemmer_es(text) -> typing.List[str]:
    stopword_es = nltk.corpus.stopwords.words('spanish')
    stemmer = SnowballStemmer("spanish")

    clean_words = [word for word in word_tokenize(text) if word not in string.punctuation and word.lower() not in stopword_es] # list[str]
    return [stemmer.stem(word) for word in clean_words]  # list[str]


stopwords_es = nltk.corpus.stopwords.words('spanish')

example = df_train.loc[0, "x_text"]
ex_stem = tokenizer_stemmer_es(example)

print(f"{example=}")
print(f"{ex_stem=}")

example='Respuestas coherentes e idénticas ante entradas de audio o texto: Los usuarios tienen la posibilidad de escuchar la respuesta mediante voz, esta ha de ser entendida e idéntica a la respuesta por escrito.'
ex_stem=['respuest', 'coherent', 'ident', 'entrad', 'audi', 'text', 'usuari', 'posibil', 'escuch', 'respuest', 'mediant', 'voz', 'ser', 'entend', 'ident', 'respuest', 'escrit']


In [7]:
tfidf_unigrams = TfidfVectorizer(
    strip_accents="ascii",
    lowercase=True,
    tokenizer=tokenizer_stemmer_es,
    ngram_range=(1, 1),
    binary=True,
)


clf = GradientBoostingClassifier(
    n_estimators=2000,  # Many boosting rounds  so early stoping takes place
    validation_fraction=0.2,  # Early stopping
    random_state=RND_SEED)

# Crear la tubería
skl_pl = Pipeline([
    ('fte', tfidf_unigrams),
    ('clf', clf)
])

# Búsqueda en Cuadrícula

Búsqueda en CuadrículaCV will run a set of Cross Validation experiments for you.
Se ejecutará para cada combinación de hiperparámetros en el `param_grid`
y ejecutar un trabajo de Validación Cruzada para cada uno.


Recuerda usar siempre el mismo número de pliegues CV y la misma métrica CV en 
¡cada experimento!


In [8]:
X_train = df_train['x_text']
y_train = df_train['y_is_nf']


param_grid = {
    'fte__max_features':[64, 128, 256],
    'fte__max_df': [0.25, 0.5, 0.95],
    'fte__min_df': [1, 3],
    'clf__max_depth': [3, 5, 7]
}

grid_search = GridSearchCV(
    skl_pl,
    param_grid,
    cv=K_FOLD,
    scoring='f1',
    n_jobs=-1
    )

# Ajustar GridSearchCV en los datos de entrenamiento
grid_search.fit(X_train, y_train)
print(f"{grid_search.best_score_=}")

c:\Users\usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


grid_search.best_score_=np.float64(0.7299351499811134)


In [9]:
df_exp_summary = pd.DataFrame(
    grid_search.cv_results_
)

df_exp_summary["experiment_id"] = EXPERIMENT
df_exp_summary.sort_values(ascending=True, by="rank_test_score").head(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf__max_depth,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
40,5.736805,0.277375,0.193339,0.059588,7,0.25,256,1,"{'clf__max_depth': 7, 'fte__max_df': 0.25, 'ft...",0.734694,0.723404,0.731707,0.729935,0.004776,1,exp02_hpt_gbt
46,5.644761,0.128629,0.310168,0.018758,7,0.50,256,1,"{'clf__max_depth': 7, 'fte__max_df': 0.5, 'fte...",0.734694,0.723404,0.731707,0.729935,0.004776,1,exp02_hpt_gbt
47,6.118877,0.720558,0.208212,0.044737,7,0.50,256,3,"{'clf__max_depth': 7, 'fte__max_df': 0.5, 'fte...",0.720000,0.695652,0.731707,0.715786,0.015018,3,exp02_hpt_gbt
41,5.813476,0.385993,0.207325,0.050984,7,0.25,256,3,"{'clf__max_depth': 7, 'fte__max_df': 0.25, 'ft...",0.720000,0.695652,0.731707,0.715786,0.015018,3,exp02_hpt_gbt
52,4.747097,0.471570,0.103352,0.011938,7,0.95,256,1,"{'clf__max_depth': 7, 'fte__max_df': 0.95, 'ft...",0.693878,0.723404,0.714286,0.710523,0.012344,5,exp02_hpt_gbt


# Diagnosticar el modelo

In [10]:
# Verificar dimensiones de DTM
skl_pl_fitted = grid_search.best_estimator_  

# Access the Vectorizer part of the pipeline
skl_pl_fte = skl_pl_fitted.named_steps['fte']

# Obtener DTM con transform()
dtm_train = skl_pl_fte.transform(X_train)
print(f"{dtm_train.shape=}")  # columnas: Número de términos en el vocabulario

dtm_train.shape=(311, 256)


In [11]:
# Verificar predicciones de entrenamiento y puntuación

y_hats_train = grid_search.best_estimator_.predict(X_train)  # obtener predicciones con predict()
f1_score_train = f1_score(
    y_true=y_train,
    y_pred=y_hats_train
)

print(f"{f1_score_train=}")  # ¿Es comparable con la métrica CV?

f1_score_train=1.0


# Escribir resultados de experimentos

In [12]:
df_exp_summary.to_csv(
    os.path.join(path_experiment, file_exp),
    index=False
)

# otros resultados de experimentos y artefactos podrían ser útiles